# 07 Supervisor Agent

Run the notebook-facing supervisor service. The supervisor calls the RAG-based fundamental worker, technical chart worker, and Tavily news worker, then aggregates their 1-100 ratings into a final future-perspective rating.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from market_analyst.config.settings import load_settings
from market_analyst.services.supervisor import run_supervisor_agent
from market_analyst.telemetry import configure_notebook_logging
from market_analyst.types.supervisor import SupervisorAnalysisRequest

logger = configure_notebook_logging(run_name="07_supervisor_agent")
settings = load_settings()

settings.require_chat_model()
settings.require_database()
settings.require_embeddings()
settings.require_tavily()

print("Project root:", PROJECT_ROOT)
print("Chat deployment:", settings.azure_openai_chat_deployment)
print("Vector collection:", settings.vector_collection_name)
print("Tavily configured:", bool(settings.tavily_api_key))

## Run Configuration

Set the sample company inputs. The RAG store should already contain annual-report chunks from `03_rag_pipeline.ipynb` or the shared backend ingestion path. The technical worker will fetch price history for the ticker, and the news worker will run current company plus sector searches.

In [ ]:
COMPANY_NAME = "Sample Company"
TICKER = "SAMPLE"
SECTOR = None

request = SupervisorAnalysisRequest(
    company_name=COMPANY_NAME,
    ticker=TICKER,
    sector=SECTOR,
)

print(request)

## Run The Supervisor

The call below uses the shared supervisor service. Worker prompts and score parsing stay in reusable modules; the notebook only configures inputs and displays the result.

In [ ]:
result = run_supervisor_agent(settings, request)

print(result.summary)

## Inspect Worker Ratings

This cell prints the worker ratings and rationales that produced the final supervisor rating.

In [ ]:
for component in result.components:
    print(f"{component.name}: rating={component.rating}, weight={component.weight:.2f}")
    print(component.rationale[:500])
    print()

## Validation

In [ ]:
assert 1 <= result.final_rating <= 100
assert len(result.components) == 3
assert {component.name for component in result.components} == {"fundamental", "technical", "news"}

print("Supervisor notebook validation passed.")

## Supervisor Chat Follow-Up Test

This section validates the chat-facing supervisor layer. It reuses the static `result` snapshot above, keeps short-term chat history in the notebook, and lets the supervisor chat agent call the fundamental, technical, or news worker tools when the user asks a follow-up question.

In [ ]:
from market_analyst.services.supervisor_chat import run_supervisor_chat_turn
from market_analyst.types.supervisor_chat import SupervisorChatContext, SupervisorChatRequest

chat_context = SupervisorChatContext(
    company_name=request.company_name,
    ticker=request.ticker,
    sector=request.sector,
    supervisor_result=result,
)

chat_history = []
print(chat_context)

## Ask A Supervisor-Level Follow-Up

This first turn should usually answer from the attached supervisor snapshot. The model can still call worker tools if it needs more evidence.

In [ ]:
chat_request = SupervisorChatRequest(
    context=chat_context,
    message="What is the main reason behind the final rating, and which area should I inspect first?",
    history=chat_history,
)

chat_response = run_supervisor_chat_turn(settings, chat_request)
chat_history = chat_response.history

print(chat_response.answer)
print("\nTools used:", chat_response.tool_names)
print("History length:", len(chat_history))

## Ask A Tool-Routed Follow-Up

This turn asks specifically for a technical view, so the supervisor chat agent should be able to call the technical worker tool while preserving the Yahoo Finance ticker exactly as provided.

In [ ]:
tool_request = SupervisorChatRequest(
    context=chat_context,
    message="Ask the technical agent whether the current momentum supports the supervisor rating.",
    history=chat_history,
)

tool_response = run_supervisor_chat_turn(settings, tool_request)
chat_history = tool_response.history

print(tool_response.answer)
print("\nTools used:", tool_response.tool_names)
print("History length:", len(chat_history))

## Chat Validation

In [ ]:
assert chat_response.answer.strip(), "Supervisor chat response should not be empty."
assert tool_response.answer.strip(), "Tool-routed supervisor chat response should not be empty."
assert len(chat_history) >= 4, "Chat history should include both user/assistant turns."
assert chat_history[-1].role == "assistant"

print("Supervisor chat notebook validation passed.")